# Phase 2: Feature Engineering

This notebook performs feature engineering for match predictor.

In [1]:
import pandas as pd
import numpy as np

# Load all data
results_df = pd.read_csv('../data/results.csv')
elo_df = pd.read_csv('../data/elo_ratings.csv')
former_names_df = pd.read_csv('../data/former_names.csv')

## 1. Standardize Team Names

Historical teams like 'Czech Republic' to 'Czechia', 'Soviet Union' to 'Russia', etc.

In [2]:
# Create name mapping
name_mapping = dict(zip(former_names_df['former'], former_names_df['current']))

def standardize_team(team_name):
    return name_mapping.get(team_name, team_name)

# Apply to results_df and elo_df
results_df['home_team'] = results_df['home_team'].apply(standardize_team)
results_df['away_team'] = results_df['away_team'].apply(standardize_team)
elo_df['team'] = elo_df['team'].apply(standardize_team)

results_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## 2. Calculate Target Variable ('result')

- 0: Home Win
- 1: Draw
- 2: Away Win

In [3]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 0
    elif row['home_score'] < row['away_score']:
        return 2
    else:
        return 1

results_df['result'] = results_df.apply(get_result, axis=1)
results_df['result'].value_counts()

result
0    24165
2    13935
1    11278
Name: count, dtype: int64

## 3. Compute Rolling 5-Match Form Features

In [4]:
team_history = {}

home_form = []
away_form = []

home_gf = []
away_gf = []

home_gc = []
away_gc = []

home_gd = []
away_gd = []

# Sort data by date to ensure correct order
results_df = results_df.sort_values('date').reset_index(drop=True)

for _, row in results_df.iterrows():
    home = row['home_team']
    away = row['away_team']

    if home not in team_history:
        team_history[home] = []
    if away not in team_history:
        team_history[away] = []

    # Get last 5 matches
    home_last5 = team_history[home][-5:]
    away_last5 = team_history[away][-5:]

    # Calculate features
    home_form.append(sum(m['p'] for m in home_last5))
    away_form.append(sum(m['p'] for m in away_last5))

    hgf = sum(m['gf'] for m in home_last5)
    agf = sum(m['gf'] for m in away_last5)
    home_gf.append(hgf)
    away_gf.append(agf)

    hgc = sum(m['gc'] for m in home_last5)
    agc = sum(m['gc'] for m in away_last5)
    home_gc.append(hgc)
    away_gc.append(agc)

    home_gd.append(hgf - hgc)
    away_gd.append(agf - agc)

    # Update history after prediction
    hp = 3 if row['home_score'] > row['away_score'] else (1 if row['home_score'] == row['away_score'] else 0)
    ap = 3 if row['away_score'] > row['home_score'] else (1 if row['home_score'] == row['away_score'] else 0)
    
    if not pd.isna(row['home_score']): # Only add actual played matches
        team_history[home].append({'p': hp, 'gf': row['home_score'], 'gc': row['away_score']})
        team_history[away].append({'p': ap, 'gf': row['away_score'], 'gc': row['home_score']})

# Assign to DataFrame
results_df['home_form_points_last5'] = home_form
results_df['away_form_points_last5'] = away_form

results_df['home_goals_scored_last5'] = home_gf
results_df['away_goals_scored_last5'] = away_gf

results_df['home_goals_conceded_last5'] = home_gc
results_df['away_goals_conceded_last5'] = away_gc

results_df['home_goal_difference_last5'] = home_gd
results_df['away_goal_difference_last5'] = away_gd

results_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,home_form_points_last5,away_form_points_last5,home_goals_scored_last5,away_goals_scored_last5,home_goals_conceded_last5,away_goals_conceded_last5,home_goal_difference_last5,away_goal_difference_last5
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,0,1,1,0.0,0.0,0.0,0.0,0.0,0.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,0,1,4,2.0,4.0,4.0,2.0,-2.0,2.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1,4,4,5.0,4.0,4.0,5.0,1.0,-1.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,0,5,5,6.0,7.0,7.0,6.0,-1.0,1.0


## 4. Merge Elo Ratings

In [5]:
# Add year
results_df['year'] = pd.to_datetime(results_df['date']).dt.year

# Merge home Elo
results_df = results_df.merge(elo_df, left_on=['year', 'home_team'], right_on=['year', 'team'], how='left')
results_df.rename(columns={'rating': 'home_elo'}, inplace=True)
results_df.drop('team', axis=1, inplace=True)

# Merge away Elo
results_df = results_df.merge(elo_df, left_on=['year', 'away_team'], right_on=['year', 'team'], how='left')
results_df.rename(columns={'rating': 'away_elo'}, inplace=True)
results_df.drop('team', axis=1, inplace=True)

# Calculate Elo difference
results_df['elo_difference'] = results_df['home_elo'] - results_df['away_elo']

print(f"Before cleaning missing Elo: {len(results_df)} rows")
results_df = results_df[results_df['year'] >= 1901]
results_df.dropna(subset=['home_elo', 'away_elo'], inplace=True)
print(f"After cleaning: {len(results_df)} rows")
results_df.head()

Before cleaning missing Elo: 49961 rows
After cleaning: 42949 rows


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,home_goals_scored_last5,away_goals_scored_last5,home_goals_conceded_last5,away_goals_conceded_last5,home_goal_difference_last5,away_goal_difference_last5,year,home_elo,away_elo,elo_difference
133,1901-02-23,Scotland,Northern Ireland,11.0,0.0,British Home Championship,Glasgow,Scotland,False,0,...,22.0,2.0,6.0,16.0,16.0,-14.0,1901,1973.0,1338.0,635.0
134,1901-03-02,Wales,Scotland,1.0,1.0,British Home Championship,Wrexham,Wales,False,1,...,5.0,24.0,16.0,5.0,-11.0,19.0,1901,1476.0,1973.0,-497.0
135,1901-03-09,England,Northern Ireland,3.0,0.0,British Home Championship,Southampton,England,False,0,...,10.0,1.0,6.0,27.0,4.0,-26.0,1901,2013.0,1338.0,675.0
136,1901-03-18,England,Wales,6.0,0.0,British Home Championship,Newcastle,England,False,0,...,9.0,6.0,6.0,11.0,3.0,-5.0,1901,2013.0,1476.0,537.0
137,1901-03-23,Northern Ireland,Wales,0.0,1.0,British Home Championship,Belfast,Ireland,False,2,...,0.0,6.0,21.0,13.0,-21.0,-7.0,1901,1338.0,1476.0,-138.0


In [6]:
# Save processed features
results_df.to_csv('../data/features_with_elo_cleaned.csv', index=False)
print('Saved: features_with_elo_cleaned.csv!')

Saved: features_with_elo_cleaned.csv!
